# 11 — Ban & Pick Meta Analysis

## Business Question
Is the community banning the right champions? What does the ban/pick meta reveal about perceived vs actual threat?

## Statistical Depth
- Ban rate vs win rate scatter with Spearman correlation
- Presence rate (ban + pick combined) as proxy for meta dominance
- Champion threat score: composite metric combining win rate, ban rate, and statistical significance
- Identifying "stealth OP" champions (high win rate, low ban rate — community underestimating threat)

In [3]:
import sys
sys.path.insert(0, '../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

from config import *
from data_loader import load_matches, load_champion_map, build_champion_stats
from stats_utils import spearman_correlation, test_win_rate, benjamini_hochberg
from plot_utils import set_style, save_plot

set_style()
df = load_matches()
champ_map = load_champion_map()
stats = build_champion_stats(df, champ_map)

# Add statistical significance
p_vals = []
for _, row in stats.iterrows():
    r = test_win_rate(int(row['wins']), int(row['games']), h0_rate=0.5)
    p_vals.append(r['p_value'])
bh_sig = benjamini_hochberg(p_vals, alpha=ALPHA)
stats['significant'] = bh_sig

# Presence rate = pick rate + ban rate (total meta impact)
stats['presence_rate'] = stats['pick_rate'] + stats['ban_rate']

# Threat score: win rate deviation * ban rate * significance multiplier
stats['win_rate_deviation'] = stats['win_rate'] - 50
stats['threat_score'] = (
    stats['win_rate_deviation'].abs() * 
    (stats['ban_rate'] / 100 + 0.1) *
    stats['significant'].map({True: 2.0, False: 0.5})
).round(3)

print(f"Champions analysed: {len(stats)}")
print(f"Champions: {stats['significant'].sum()} significantly different from 50% win rate")
print("\nTop 10 threat scores:")
print(stats.nlargest(10, 'threat_score')[['champion','win_rate','ban_rate','threat_score','significant']].to_string(index=False))
print("\nStealth OP (high win rate, low ban rate, significant):")
stealth = stats[(stats['win_rate'] > WIN_RATE_OP) & (stats['ban_rate'] < 10) & stats['significant']]
print(stealth[['champion','win_rate','ban_rate','games']].sort_values('win_rate', ascending=False).to_string(index=False))

Champions analysed: 138
Champions: 46 significantly different from 50% win rate

Top 10 threat scores:
champion  win_rate  ban_rate  threat_score  significant
   Janna     55.53     41.54         5.700         True
    Ornn     41.02     11.04         3.779         True
Cho'Gath     52.67     48.89         3.145         True
  Twitch     52.78     30.75         2.266         True
    Ryze     40.79      0.19         1.877         True
 Lee Sin     45.90     12.65         1.857         True
 Caitlyn     46.97     19.90         1.812         True
    Azir     43.36      0.81         1.436         True
  Syndra     46.57      8.82         1.291         True
 Sejuani     53.12     10.16         1.258         True

Stealth OP (high win rate, low ban rate, significant):
champion  win_rate  ban_rate  games
    Sona     54.19      1.19   5429
  Yorick     53.99      0.98   1378
  Rammus     53.85      3.59   2997
  Anivia     53.60      1.70   2252
  Singed     53.47      0.98   1425
   Swain 

In [4]:
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# 1. Ban rate vs win rate scatter
scatter = axes[0,0].scatter(
    stats['ban_rate'], stats['win_rate'],
    s=stats['games'] / 20,
    c=stats['win_rate'],
    cmap='RdYlGn', vmin=44, vmax=57,
    alpha=0.75, edgecolors='white', linewidth=0.5
)
axes[0,0].axhline(50, color=COLORS['gray'], linestyle='--', linewidth=1.5)
axes[0,0].axvline(BAN_RATE_HIGH, color=COLORS['gray'], linestyle='--', linewidth=1.5)
for _, row in stats[(stats['ban_rate'] > 30) | (stats['win_rate'] > 54) | (stats['win_rate'] < 46)].iterrows():
    axes[0,0].annotate(row['champion'], (row['ban_rate'], row['win_rate']),
        fontsize=7, ha='center', va='bottom', xytext=(0, 3), textcoords='offset points')
plt.colorbar(scatter, ax=axes[0,0], label='Win Rate (%)')
axes[0,0].set_xlabel('Ban Rate (%)')
axes[0,0].set_ylabel('Win Rate (%)')
axes[0,0].set_title('Ban Rate vs Win Rate Scatter\n(bubble size = games played)')

# 2. Threat score ranking
top_threat = stats.nlargest(15, 'threat_score').sort_values('threat_score', ascending=True)
colors_t = [COLORS['red'] if r['win_rate'] > 50 else COLORS['blue'] for _, r in top_threat.iterrows()]
bars = axes[0,1].barh(top_threat['champion'], top_threat['threat_score'],
                      color=colors_t, edgecolor='white', height=0.7)
for bar, val in zip(bars, top_threat['threat_score']):
    axes[0,1].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                  f'{val:.2f}', va='center', fontsize=9)
axes[0,1].set_xlabel('Threat Score (composite metric)')
axes[0,1].set_title('Top 15 Champions by Threat Score\n(Red=OP, Blue=UP)')

# 3. Presence rate
top_presence = stats.nlargest(15, 'presence_rate').sort_values('presence_rate', ascending=True)
axes[1,0].barh(top_presence['champion'], top_presence['ban_rate'],
               color=COLORS['red'], label='Ban Rate', edgecolor='white', height=0.7)
axes[1,0].barh(top_presence['champion'], top_presence['pick_rate'],
               left=top_presence['ban_rate'], color=COLORS['blue'],
               label='Pick Rate', edgecolor='white', height=0.7, alpha=0.8)
axes[1,0].set_xlabel('Presence Rate (Ban + Pick %)')
axes[1,0].set_title('Champion Presence Rate\n(Total meta impact = ban + pick)')
axes[1,0].legend()

# 4. Stealth OP / Overrated analysis
stealth_op = stats[(stats['win_rate'] > WIN_RATE_OP) & (stats['ban_rate'] < 10)]
overrated  = stats[(stats['win_rate'] < 51) & (stats['ban_rate'] > BAN_RATE_HIGH)]

all_cats = pd.concat([
    stealth_op.assign(category='Stealth OP\n(High WR, Low Ban)'),
    overrated.assign(category='Overrated\n(Low WR, High Ban)')
])

if len(all_cats) > 0:
    axes[1,1].scatter(all_cats['ban_rate'], all_cats['win_rate'],
                      c=all_cats['category'].map({
                          'Stealth OP\n(High WR, Low Ban)': COLORS['orange'],
                          'Overrated\n(Low WR, High Ban)': COLORS['blue']
                      }),
                      s=100, alpha=0.8, edgecolors='black', linewidth=0.5)
    for _, row in all_cats.iterrows():
        axes[1,1].annotate(row['champion'],
            (row['ban_rate'], row['win_rate']),
            fontsize=8, ha='center', va='bottom', xytext=(0, 4), textcoords='offset points')

axes[1,1].axhline(50, color=COLORS['gray'], linestyle='--', linewidth=1.5)
axes[1,1].axvline(BAN_RATE_HIGH, color=COLORS['gray'], linestyle='--', linewidth=1.5)
axes[1,1].set_xlabel('Ban Rate (%)')
axes[1,1].set_ylabel('Win Rate (%)')
axes[1,1].set_title('Stealth OP vs Overrated Champions')

plt.suptitle('Ban & Pick Meta Analysis — Season 9', fontsize=14, fontweight='bold')
save_plot('11_ban_pick_meta.png')
plt.show()

  Saved -> plots/11_ban_pick_meta.png


## Summary

**Threat Score Methodology:** The composite threat score combines win rate deviation from 50%, ban rate (community perception), and statistical significance — giving a single number that reflects both actual strength and community impact.

**Stealth OP:** Champions with high win rate but low ban rate are undervalued by the community — these are the picks that analysts should flag for the balance team *before* the community discovers them and they become meta-defining.

**Overrated:** Champions with high ban rates despite average win rates represent community perception failures — these bans waste resources on non-threats.